# Prototype XGBoost Training

This notebook trains a first-pass XGBoost model on the `v3` feature dataset.

Design choices:
- Uses the exported `train.parquet` and `val.parquet` splits from `05_feature_engineering_v2.ipynb`.
- Treats the task as binary classification per `(track_id, target_country)` pair.
- Evaluates the model as a ranking problem by sorting countries within each track.
- Samples whole tracks by default so the notebook stays lightweight for prototyping.


In [5]:
from pathlib import Path
import json
import warnings

import duckdb
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score

try:
    import xgboost as xgb
except ImportError as exc:
    raise ImportError("Install xgboost first, e.g. `pip install -r requirements.txt`.") from exc

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "datasets").exists() and (candidate / "requirements.txt").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root from {start}. Expected a parent containing 'datasets/' and 'requirements.txt'."
    )

ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = ROOT / "datasets" / "processed" / "v3_features"
ARTIFACT_DIR = ROOT / "artifacts" / "models" / "xgboost_prototype"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.parquet"
VAL_PATH = DATA_DIR / "val.parquet"

assert TRAIN_PATH.exists(), f"Missing training split: {TRAIN_PATH}"
assert VAL_PATH.exists(), f"Missing validation split: {VAL_PATH}"

RANDOM_STATE = 42
TOP_K = 5
TRAIN_MAX_TRACKS = 30000  # set to None to use the full training split
VAL_MAX_TRACKS = 10000    # set to None to use the full validation split

FEATURE_EXCLUDE = [
    "track_id",
    "observation_time",
    "target_country",
    "did_enter_within_60d",
    "days_to_entry",
]


In [6]:
con = duckdb.connect()

def load_split(path: Path, max_tracks: int | None = None) -> pd.DataFrame:
    parquet_path = path.as_posix()
    if max_tracks is None:
        query = f"SELECT * FROM read_parquet('{parquet_path}')"
    else:
        query = f"""
            WITH sampled_tracks AS (
                SELECT track_id
                FROM read_parquet('{parquet_path}')
                GROUP BY track_id
                ORDER BY hash(track_id)
                LIMIT {int(max_tracks)}
            )
            SELECT d.*
            FROM read_parquet('{parquet_path}') d
            JOIN sampled_tracks st USING (track_id)
        """
    return con.execute(query).fetchdf()

def make_feature_matrix(df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series) -> pd.DataFrame:
    X = df[feature_cols].copy()
    return X.fillna(fill_values)

def ranking_metrics(scored_df: pd.DataFrame, k: int = 5) -> dict:
    rows = []
    for track_id, group in scored_df.groupby("track_id", sort=False):
        group = group.sort_values("score", ascending=False).reset_index(drop=True)
        labels = group["did_enter_within_60d"].to_numpy(dtype=int)
        positives = int(labels.sum())
        top = group.head(k)
        top_labels = top["did_enter_within_60d"].to_numpy(dtype=int)
        hits = int(top_labels.sum())

        precision = hits / k
        recall = hits / positives if positives else np.nan
        hit_rate = float(hits > 0) if positives else np.nan

        discounts = np.log2(np.arange(2, len(top_labels) + 2))
        dcg = float(((2 ** top_labels - 1) / discounts).sum())
        ideal = np.sort(labels)[::-1][: len(top_labels)]
        idcg = float(((2 ** ideal - 1) / np.log2(np.arange(2, len(ideal) + 2))).sum())
        ndcg = dcg / idcg if idcg > 0 else np.nan

        ap_accum = 0.0
        running_hits = 0
        for rank, rel in enumerate(top_labels, start=1):
            if rel:
                running_hits += 1
                ap_accum += running_hits / rank
        map_k = ap_accum / min(positives, k) if positives else np.nan

        rows.append(
            {
                "track_id": track_id,
                "positives": positives,
                f"precision@{k}": precision,
                f"recall@{k}": recall,
                f"hit_rate@{k}": hit_rate,
                f"ndcg@{k}": ndcg,
                f"map@{k}": map_k,
            }
        )

    metric_df = pd.DataFrame(rows)
    positive_mask = metric_df["positives"] > 0
    summary = {
        "tracks": int(metric_df.shape[0]),
        "positive_tracks": int(positive_mask.sum()),
        f"precision@{k}": float(metric_df[f"precision@{k}"].mean()),
        f"recall@{k}": float(metric_df.loc[positive_mask, f"recall@{k}"].mean()) if positive_mask.any() else None,
        f"hit_rate@{k}": float(metric_df.loc[positive_mask, f"hit_rate@{k}"].mean()) if positive_mask.any() else None,
        f"ndcg@{k}": float(metric_df.loc[positive_mask, f"ndcg@{k}"].mean()) if positive_mask.any() else None,
        f"map@{k}": float(metric_df.loc[positive_mask, f"map@{k}"].mean()) if positive_mask.any() else None,
        "mean_future_countries_per_track": float(metric_df["positives"].mean()),
    }
    return summary

def binary_metrics(y_true: pd.Series, y_score: np.ndarray) -> dict:
    return {
        "roc_auc": float(roc_auc_score(y_true, y_score)),
        "average_precision": float(average_precision_score(y_true, y_score)),
        "log_loss": float(log_loss(y_true, y_score, labels=[0, 1])),
    }


In [7]:
train_df = load_split(TRAIN_PATH, TRAIN_MAX_TRACKS)
val_df = load_split(VAL_PATH, VAL_MAX_TRACKS)

feature_cols = [col for col in train_df.columns if col not in FEATURE_EXCLUDE]
fill_values = train_df[feature_cols].median(numeric_only=True)

X_train = make_feature_matrix(train_df, feature_cols, fill_values)
X_val = make_feature_matrix(val_df, feature_cols, fill_values)
y_train = train_df["did_enter_within_60d"].astype(int)
y_val = val_df["did_enter_within_60d"].astype(int)

negatives = int((y_train == 0).sum())
positives = int((y_train == 1).sum())
scale_pos_weight = negatives / max(positives, 1)

print(f"Train rows: {len(train_df):,} | tracks: {train_df['track_id'].nunique():,}")
print(f"Val rows:   {len(val_df):,} | tracks: {val_df['track_id'].nunique():,}")
print(f"Feature count: {len(feature_cols)}")
print(f"Training positives: {positives:,} ({positives / len(train_df) * 100:.2f}%)")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

X_train.head(3)


Train rows: 127,940 | tracks: 30,000
Val rows:   636,230 | tracks: 10,000
Feature count: 103
Training positives: 21,167 (16.54%)
scale_pos_weight: 5.04


,rank_andorra,rank_argentina,rank_australia,rank_austria,rank_belgium,rank_bolivia,rank_brazil,rank_bulgaria,rank_canada,rank_chile,rank_colombia,rank_costa_rica,rank_czech_republic,rank_denmark,rank_dominican_republic,rank_ecuador,rank_egypt,rank_el_salvador,rank_estonia,rank_finland,rank_france,rank_germany,rank_greece,rank_guatemala,rank_honduras,rank_hong_kong,rank_hungary,rank_iceland,rank_india,rank_indonesia,rank_ireland,rank_israel,rank_italy,rank_japan,rank_latvia,rank_lithuania,rank_luxembourg,rank_malaysia,rank_mexico,rank_morocco,rank_netherlands,rank_new_zealand,rank_nicaragua,rank_norway,rank_panama,rank_paraguay,rank_peru,rank_philippines,rank_poland,rank_portugal,rank_romania,rank_saudi_arabia,rank_singapore,rank_slovakia,rank_south_africa,rank_spain,rank_sweden,rank_switzerland,rank_taiwan,rank_thailand,rank_turkey,rank_united_arab_emirates,rank_united_kingdom,rank_united_states,rank_uruguay,rank_vietnam,af_danceability,af_energy,af_valence,af_tempo,af_acousticness,af_speechiness,af_instrumentalness,af_liveness,duration_ms,explicit,days_since_release,is_friday_release,track_in_viral50_at_obs,artist_prior_chart_count,artist_prior_unique_regions,artist_prior_best_rank,artist_prior_unique_tracks,multi_artist_flag,artist_country_ratio,artist_prior_success_in_target,target_population,target_avg_daily_streams,target_new_entry_rate_30d,target_continent_africa,target_continent_antarctica,target_continent_asia,target_continent_europe,target_continent_north_america,target_continent_oceania,target_continent_south_america,cultural_dist_min,cultural_dist_missing,same_language_flag,same_continent_flag,neighbor_entered_count,observation_month,observation_year
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,114,0,0,0,0,0,0,0,0,0,0,0,0.771,0.586,0.408,103.057,0.217,0.0480,0.000121,0.2350,214533,0,0,0,0,1,1,188,1,0,1.000000,0,5831404,18829.184167,0.050833,0,0,0,1,0,0,0,1.620,1,0,0,0.0,10,2019
1,0,0,0,0,0,0,55,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.590,0.846,0.565,167.991,0.108,0.0616,0.000000,0.9300,194580,0,0,0,0,4998,2,3,30,1,0.066667,0,5831404,21431.727167,0.080833,0,0,0,1,0,0,0,4.423,0,0,0,0.0,12,2019
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,106,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.517,0.809,0.748,75.251,0.233,0.2200,0.000155,0.0509,151456,1,1,1,0,0,0,201,0,1,0.000000,0,5831404,18348.909833,0.041333,0,0,0,1,0,0,0,0.622,0,0,1,1.0,8,2019


In [8]:
model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric=["logloss", "aucpr"],
    tree_method="hist",
    n_estimators=800,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
    early_stopping_rounds=50,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

best_iteration = getattr(model, "best_iteration", None)
best_score = getattr(model, "best_score", None)
print(f"Best iteration: {best_iteration}")
print(f"Best validation score: {best_score}")


[0]	validation_0-logloss:0.66374	validation_0-aucpr:0.12637
[50]	validation_0-logloss:0.24251	validation_0-aucpr:0.21646
[100]	validation_0-logloss:0.17862	validation_0-aucpr:0.23134
[150]	validation_0-logloss:0.14933	validation_0-aucpr:0.23116
[181]	validation_0-logloss:0.13690	validation_0-aucpr:0.22747
Best iteration: 131
Best validation score: 0.23347247216386666


In [9]:
val_scores = model.predict_proba(X_val)[:, 1]

val_scored = val_df[["track_id", "observation_time", "target_country", "did_enter_within_60d", "days_to_entry"]].copy()
val_scored["score"] = val_scores

val_binary = binary_metrics(y_val, val_scores)
val_ranking = ranking_metrics(val_scored, k=TOP_K)

summary = {
    "data": {
        "train_path": str(TRAIN_PATH),
        "val_path": str(VAL_PATH),
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "train_tracks": int(train_df["track_id"].nunique()),
        "val_tracks": int(val_df["track_id"].nunique()),
        "train_max_tracks": TRAIN_MAX_TRACKS,
        "val_max_tracks": VAL_MAX_TRACKS,
    },
    "model": {
        "type": "xgboost.XGBClassifier",
        "top_k": TOP_K,
        "best_iteration": None if best_iteration is None else int(best_iteration),
        "scale_pos_weight": float(scale_pos_weight),
        "feature_count": len(feature_cols),
    },
    "validation": {
        "binary": val_binary,
        "ranking": val_ranking,
    },
    "feature_cols": feature_cols,
    "train_fill_values": {col: float(fill_values[col]) for col in feature_cols},
}

model_path = ARTIFACT_DIR / "xgb_classifier.json"
metadata_path = ARTIFACT_DIR / "training_summary.json"
val_preds_path = ARTIFACT_DIR / "val_predictions.parquet"

model.save_model(model_path)
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
con.register("val_scored_df", val_scored)
con.execute(f"COPY val_scored_df TO '{val_preds_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.unregister("val_scored_df")

print(json.dumps(summary["validation"], indent=2))
print(f"Saved model to: {model_path}")
print(f"Saved metadata to: {metadata_path}")
print(f"Saved validation predictions to: {val_preds_path}")

pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(20).to_frame("importance")


{
  "binary": {
    "roc_auc": 0.9542507400887033,
    "average_precision": 0.23357310088500333,
    "log_loss": 0.15842729911801587
  },
  "ranking": {
    "tracks": 10000,
    "positive_tracks": 811,
    "precision@5": 0.027520000000000006,
    "recall@5": 0.5989744937773717,
    "hit_rate@5": 0.8310727496917386,
    "ndcg@5": 0.6390557231427955,
    "map@5": 0.5781291957802438,
    "mean_future_countries_per_track": 0.4038
  }
}
Saved model to: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/artifacts/models/xgboost_prototype/xgb_classifier.json
Saved metadata to: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/artifacts/models/xgboost_prototype/training_summary.json
Saved validation predictions to: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/artifacts/models/xgboost_prototype/val_predictions.parquet


,importance
artist_prior_success_in_target,0.136478
same_language_flag,0.128389
rank_united_states,0.040248
track_in_viral50_at_obs,0.035514
same_continent_flag,0.028250
artist_country_ratio,0.023333
artist_prior_best_rank,0.020353
rank_canada,0.019906
rank_poland,0.017023
rank_honduras,0.016868
